## 1. Reconstrucción de la tabla completa

La Fase 1 dejó la unión de las 46 hojas por número de bloque solo en el notebook del EDA, sin guardarla como archivo. Para que el preprocesamiento sea reproducible desde los Excel originales, esa unión se rehace con la función `construir_tabla_completa` de `src/preprocessing_pipeline.py`, y se controla que coincida con la del EDA.


In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

sys.path.append("../src")
from auxiliar_functions import tabla_simple
from preprocessing_pipeline import construir_tabla_completa

import importlib
import preprocessing_pipeline

importlib.reload(preprocessing_pipeline)    # vuelve a leer el archivo de src/


<module 'preprocessing_pipeline' from 'c:\\Users\\Asus\\Desktop\\Automatizaciones\\Materias Facultad\\Tercer Año - Segundo Cuatrimestre\\Laboratorio de Datos II\\Trabajo Práctico Final LD2\\notebooks\\../src\\preprocessing_pipeline.py'>

In [2]:
from preprocessing_pipeline import cargar_tabla_completa
tabla_completa = cargar_tabla_completa(Path("../data"))   # la primera vez tarda ~6 min; después, segundos

control = pd.DataFrame({"Valor": {
    "Días (filas)": len(tabla_completa),
    "Columnas": tabla_completa.shape[1],
    "Días válidos": int(tabla_completa["valido"].sum()),
    "Filas con objetivo": int(tabla_completa["objetivo"].notna().sum()),
    "Columnas de las otras hojas": int(sum(" | " in str(c) for c in tabla_completa.columns)),
}})
print("Control de la tabla completa (debe coincidir con el EDA: 1.212 días, 453 columnas, 1.187 válidos, 1.186 con objetivo, 420 de otras hojas)")
display(tabla_simple(control, precision=0))


Control de la tabla completa (debe coincidir con el EDA: 1.212 días, 453 columnas, 1.187 válidos, 1.186 con objetivo, 420 de otras hojas)


,Valor
Días (filas),1212
Columnas,453
Días válidos,1187
Filas con objetivo,1186
Columnas de las otras hojas,420


## 2. División temporal en entrenamiento, validación y prueba

Antes de tomar cualquier decisión de limpieza o de selección se divide la tabla por orden de fecha, sin mezclar las filas: el 70 % más antiguo (848 días, hasta el 28/10/2022) para entrenamiento, el 15 % siguiente (182 días) para validación y el 15 % más reciente (182 días, desde el 28/04/2023) para prueba. Mezclar las filas, como hace `train_test_split`, dejaría días futuros dentro del entrenamiento; con este corte, cada criterio se mide solo con entrenamiento y la prueba se toca una única vez, al final. El corte de entrenamiento coincide con el que se usó en el EDA, así que las correlaciones y los criterios ya calculados siguen valiendo. Como el objetivo de cada día es el consumo del siguiente, el último día de cada tramo comparte una fila con el primero del tramo que sigue, un solapamiento de efecto despreciable.


In [3]:
from preprocessing_pipeline import dividir_por_fecha

X_train, X_val, X_test = dividir_por_fecha(tabla_completa)

resumen = pd.DataFrame({
    nombre: {"Días": len(parte),
             "Desde": parte["DIA"].iloc[0].strftime("%d/%m/%Y"),
             "Hasta": parte["DIA"].iloc[-1].strftime("%d/%m/%Y"),
             "Válidos con objetivo": int((parte["valido"] & parte["objetivo"].notna()).sum())}
    for nombre, parte in {"X_train": X_train, "X_val": X_val, "X_test": X_test}.items()}).T
print("División temporal de la tabla completa (70 % / 15 % / 15 %, sin mezclar las filas)")
display(tabla_simple(resumen, precision=0))


División temporal de la tabla completa (70 % / 15 % / 15 %, sin mezclar las filas)


,Días,Desde,Hasta,Válidos con objetivo
X_train,848,01/07/2020,28/10/2022,817
X_val,182,29/10/2022,27/04/2023,172
X_test,182,28/04/2023,26/10/2023,176


## 3. Limpieza del dataset, paso 1: columnas descartadas por ADR 0006

In [4]:
from auxiliar_functions import marcar_columnas_descartables, perfilar_columnas, UMBRAL_MUCHOS_FALTANTES, UMBRAL_CASI_CONSTANTE

# Columnas que no son candidatas a variable: fecha, hora, controles del día y el objetivo
COLUMNAS_DE_CONTROL = ["DIA", "HORA", "bloque", "filas", "archivo", "completo", "plausible",
                       "lectura_inicial", "mayor_incremento", "menor_incremento",
                       "tipo_falla", "valido", "consumo_dia", "objetivo"]
candidatas = [c for c in X_train.columns if c not in COLUMNAS_DE_CONTROL]

# El perfil se mide SOLO con entrenamiento
perfil = marcar_columnas_descartables(perfilar_columnas(X_train[candidatas]))
columnas_conservadas = [c for c, se_conserva in zip(candidatas, perfil["Se conserva"]) if se_conserva]

print("Resultado de los criterios del ADR 0006, medidos solo con X_train")
resumen_columnas = pd.DataFrame({"Columnas": {
    "Candidatas": len(perfil),
    "Con más de 40 % de faltantes": int(perfil["Muchos faltantes"].sum()),
    "Casi constantes (95 % o más igual)": int(perfil["Casi constante"].sum()),
    "Se conservan": len(columnas_conservadas)}})
display(tabla_simple(resumen_columnas, precision=0))

# La misma lista se aplica a las tres partes (las columnas de control se mantienen)
columnas_a_mantener = [c for c in X_train.columns if c in COLUMNAS_DE_CONTROL or c in columnas_conservadas]
X_train = X_train[columnas_a_mantener].copy()
X_val = X_val[columnas_a_mantener].copy()
X_test = X_test[columnas_a_mantener].copy()

print("Forma de cada parte después de descartar columnas (filas, columnas)")
display(tabla_simple(pd.DataFrame({"Filas": {n: p.shape[0] for n, p in {"X_train": X_train, "X_val": X_val, "X_test": X_test}.items()},
                                   "Columnas": {n: p.shape[1] for n, p in {"X_train": X_train, "X_val": X_val, "X_test": X_test}.items()}}), precision=0))


Resultado de los criterios del ADR 0006, medidos solo con X_train


,Columnas
Candidatas,439
Con más de 40 % de faltantes,142
Casi constantes (95 % o más igual),95
Se conservan,262


Forma de cada parte después de descartar columnas (filas, columnas)


,Filas,Columnas
X_train,848,276
X_val,182,276
X_test,182,276


### Paso 2: días inválidos

Los días marcados como inválidos (17 en `X_train`, 5 en `X_val` y 3 en `X_test`; ADR 0004 y 0005) se detectaron mirando solo el contador de `Frio (Kw)` de Consolidado EE. Para decidir qué columnas se anulan en esos días se comparó, solo con `X_train`, cuántos valores caen fuera del rango normal (percentiles 1 a 99 de los días válidos): Consolidado EE llega a 10,5 % en días inválidos contra 2,1 % en días válidos, mientras que el resto de las hojas se parece a lo normal (entre 0 % y 5,9 %, con pocos días como base).

**Decisión:** en los días inválidos se ponen en `NaN` las 19 columnas de Consolidado EE y las 18 razones "EE ... / Hl" de Consolidado KPI (derivadas de esos mismos kW). Las demás columnas se conservan; si alguna tiene valores absurdos, se trata en el paso siguiente. No se imputa nada y las filas se conservan (ADR 0004). El detalle queda en el ADR 0008.
La justificación que pensamos es la siguiente: si en un día el sensor falló o cortó antes, entonces las otras variables derivadas de datos medidos por ese sensor también son poco confiables.

In [5]:
# Columnas derivadas del contador de energía que falló: Consolidado EE y las razones "EE ... / Hl" del KPI
columnas_de_energia = [c for c in X_train.columns
                       if c not in COLUMNAS_DE_CONTROL
                       and (" | " not in c or c.startswith("Consolidado KPI | EE"))]

def anular_dias_invalidos(parte, columnas):
    """Pone vacío (NaN) las columnas indicadas en los días marcados como no válidos."""
    parte = parte.copy()
    parte.loc[~parte["valido"], columnas] = float("nan")
    return parte

X_train = anular_dias_invalidos(X_train, columnas_de_energia)
X_val = anular_dias_invalidos(X_val, columnas_de_energia)
X_test = anular_dias_invalidos(X_test, columnas_de_energia)

print(f"Columnas anuladas en los días inválidos: {len(columnas_de_energia)}")


Columnas anuladas en los días inválidos: 37


### Paso 3: valores absurdos e irreales

Se anulan con 'NaN' las celdas cuyos valores no pueden ser reales. No se corrigen ni se recortan, porque un valor recortado a un percentil parecería un dato verdadero y no lo es (mismo criterio que el ADR 0004).

**Regla** (límites calculados solo con `X_train` y aplicados igual a `X_val` y `X_test`). Una celda es absurda si:
1. es **negativa**, salvo en las columnas de temperatura (donde un valor negativo es posible), o
2. supera **5 veces el percentil 99** de su columna. Si ese percentil es 0 (columnas intermitentes, como los hectolitros de una marca que se produce pocos días), no se aplica límite.

**Por qué 5 veces:** el consumo de `Frio` varía 2,2 veces entre enero y julio, así que ninguna variación estacional real se acerca a ese factor. Entre 5 y 10 veces el percentil 99 hay 75 celdas en `X_train`; de las 66 que se pudieron comparar con los 3 días anteriores y los 3 posteriores, todas son picos aislados (de 7,6 a 1.890 veces la mediana de sus vecinos), no niveles altos sostenidos. Por encima de 10 veces se cumple lo mismo (132 de 138).

**Qué no se hace:** los valores altos pero creíbles (entre el percentil 99 y 5 veces ese valor) se dejan como están; si hace falta winsorizarlos se decide al elegir el modelo. Tampoco eliminamos filas: una fila con una sola celda absurda conserva sus otras columnas, y los días no se borran para no romper los rezagos del paso de variables nuevas. Detalle y alternativas descartadas en el ADR 0008.


In [6]:
columnas_de_medicion = [c for c in X_train.columns if c not in COLUMNAS_DE_CONTROL]   # las 262 columnas que no son de control

FACTOR_SOBRE_P99 = 5     # una celda es absurda si supera 5 veces el percentil 99 de su columna (medido en X_train)
limite_superior = FACTOR_SOBRE_P99 * X_train[columnas_de_medicion].apply(pd.to_numeric, errors="coerce").quantile(0.99)
limite_superior = limite_superior.where(limite_superior > 0)     # percentil 99 igual a 0: sin límite (columna intermitente)
columnas_de_temperatura = [c for c in columnas_de_medicion if "Temp" in c]

def marcar_valores_absurdos(parte, columnas, limite_superior, columnas_de_temperatura):
    """Devuelve dos tablas de verdadero/falso: celdas negativas (salvo temperaturas) y celdas por encima del límite."""
    valores = parte[columnas].apply(pd.to_numeric, errors="coerce")
    negativas = (valores < 0) & ~valores.columns.isin(columnas_de_temperatura)
    muy_altas = valores > limite_superior                    # comparar con NaN da falso: no se marca
    return negativas, muy_altas

def anular_valores_absurdos(parte, columnas, limite_superior, columnas_de_temperatura):
    """Pone vacío (NaN) las celdas marcadas como absurdas; el resto de la tabla queda igual."""
    parte = parte.copy()
    negativas, muy_altas = marcar_valores_absurdos(parte, columnas, limite_superior, columnas_de_temperatura)
    parte[columnas] = parte[columnas].apply(pd.to_numeric, errors="coerce").mask(negativas | muy_altas)
    return parte

# Primero se cuenta (antes de anular), después se anula
marcas = {n: marcar_valores_absurdos(p, columnas_de_medicion, limite_superior, columnas_de_temperatura)
          for n, p in {"X_train": X_train, "X_val": X_val, "X_test": X_test}.items()}

X_train = anular_valores_absurdos(X_train, columnas_de_medicion, limite_superior, columnas_de_temperatura)
X_val = anular_valores_absurdos(X_val, columnas_de_medicion, limite_superior, columnas_de_temperatura)
X_test = anular_valores_absurdos(X_test, columnas_de_medicion, limite_superior, columnas_de_temperatura)

print("Celdas anuladas por ser absurdas, por parte")
display(tabla_simple(pd.DataFrame({"Celdas anuladas": {n: int(negativas.sum().sum() + altos.sum().sum()) for n, (negativas, altos) in marcas.items()}}), precision=0))

anuladas_por_columna = pd.DataFrame({
    "Negativas (X_train)": marcas["X_train"][0].sum(),
    "Muy altas (X_train)": marcas["X_train"][1].sum(),
    "X_val": marcas["X_val"][0].sum() + marcas["X_val"][1].sum(),
    "X_test": marcas["X_test"][0].sum() + marcas["X_test"][1].sum()})
anuladas_por_columna = anuladas_por_columna[anuladas_por_columna.sum(axis=1) > 0]       # solo las columnas con alguna celda anulada
anuladas_por_columna = anuladas_por_columna.assign(Total=anuladas_por_columna.sum(axis=1)).sort_values("Total", ascending=False)
anuladas_por_columna.insert(0, "Hoja", [c.split(" | ")[0] if " | " in c else "Consolidado EE" for c in anuladas_por_columna.index])

columnas_train = int(((anuladas_por_columna["Negativas (X_train)"] + anuladas_por_columna["Muy altas (X_train)"]) > 0).sum())
print(f"Columnas con celdas anuladas: {len(anuladas_por_columna)} de {len(columnas_de_medicion)} en alguna de las tres partes "
      f"({columnas_train} en X_train). Se muestran las 15 con más celdas anuladas.")
display(tabla_simple(anuladas_por_columna.head(15).rename_axis("Columna").reset_index(), precision=0))


Celdas anuladas por ser absurdas, por parte


,Celdas anuladas
X_train,389
X_val,46
X_test,30


Columnas con celdas anuladas: 96 de 262 en alguna de las tres partes (91 en X_train). Se muestran las 15 con más celdas anuladas.


,Columna,Hoja,Negativas (X_train),Muy altas (X_train),X_val,X_test,Total
0,Resto Serv (Kw),Consolidado EE,25,2,8,1,36
1,Consolidado KPI | EE Resto Serv / Hl,Consolidado KPI,23,4,8,1,36
2,Consolidado KPI | Aire Servicios / Hl,Consolidado KPI,25,3,0,0,28
3,Consolidado Aire | Aire Servicios (M3),Consolidado Aire,25,0,0,0,25
4,Restos Planta (Kw),Consolidado EE,14,2,1,0,17
5,Consolidado KPI | EE Resto Planta / Hl,Consolidado KPI,12,3,1,0,16
6,Consolidado KPI | EE Agua / Hl,Consolidado KPI,5,4,0,0,9
7,Consolidado KPI | Aire L4 / Hl,Consolidado KPI,0,3,3,3,9
8,Consolidado KPI | ET Linea 4/Hl,Consolidado KPI,0,2,4,3,9
9,Consolidado KPI | EE Linea 4 / Hl,Consolidado KPI,0,3,4,1,8


### Paso 4: valores faltantes que quedan

Después de descartar columnas (paso 1), anular los días inválidos (paso 2) y los valores absurdos (paso 3), se mide cuántos vacíos quedan, por hoja y por columna, en cada parte. Es solo un diagnóstico: **este paso no modifica la tabla**.

**Qué se encontró (medido en `X_train`):**
- Faltan 2,4 % de las celdas de medición. 234 de las 262 columnas tienen 5 % o menos de vacíos; 21 tienen entre 5 % y 20 %; 7 superan 20 % (los silos de bagazo, los cocimientos diarios y las razones por hectolitro de la Línea 5).
- Por hoja, las que más vacíos tienen son Totalizadores Glicol (8,6 %), Consolidado KPI (6,9 %) y Totalizadores Producción (6,3 %).
- La mayoría de los vacíos son cortos: 204 columnas nunca tienen más de 3 días seguidos sin dato, y 2.733 de los 5.259 vacíos de entrenamiento están en tramos de 3 días o menos. Solo 5 columnas tienen tramos de más de 30 días sin dato.
- En validación y prueba las razones del KPI de la Línea 5 faltan en 61 % y 52 % de los días (contra 35 % en entrenamiento): esas columnas se están dejando de medir.

**Decisión:** no se rellena ningún vacío en la limpieza. Los vacíos se dejan como `NaN` y su tratamiento se define una sola vez, con la misma regla para todos los modelos, dentro de un `Pipeline` ajustado solo con entrenamiento (cierre de la Fase 2).

**Por qué:**
1. Rellenar con la mediana, el promedio o el valor vecino ahora usaría información de todo el período. La mediana de `X_train` solo puede calcularse dentro de cada tramo de validación temporal, no antes.
2. Los rezagos y promedios móviles de la etapa siguiente se calculan sobre días consecutivos. Un relleno previo les inventaría un dato que nunca existió.
3. Es una porción chica de la tabla (2,4 %), así que no se justifica perder la señal de "acá no hubo dato".
4. Para comparar modelos hace falta que todos reciban el mismo tratamiento (ver ADR 0008).

**Qué queda para la selección:** las columnas cuya disponibilidad cae en validación y prueba (por ejemplo, las razones de la Línea 5) se revisan al elegir variables; una variable que ya no se mide no sirve para predecir el futuro.


In [7]:
def racha_maxima_de_vacios(columna):
    """Mayor cantidad de días seguidos sin dato en una columna."""
    vacios = columna.isna()
    tramo = (vacios != vacios.shift()).cumsum()                 # un número nuevo cada vez que cambia entre dato y vacío
    return int((vacios.groupby(tramo).transform("sum") * vacios).max())   # largo del tramo vacío más largo

faltantes = pd.DataFrame({
    "% faltante X_train": 100 * X_train[columnas_de_medicion].isna().mean(),
    "% faltante X_val": 100 * X_val[columnas_de_medicion].isna().mean(),
    "% faltante X_test": 100 * X_test[columnas_de_medicion].isna().mean(),
    "Racha máxima X_train (días)": X_train[columnas_de_medicion].apply(racha_maxima_de_vacios)})
faltantes.insert(0, "Hoja", [c.split(" | ")[0] if " | " in c else "Consolidado EE" for c in faltantes.index])

celdas = X_train[columnas_de_medicion].size
vacias = int(X_train[columnas_de_medicion].isna().sum().sum())
print(f"X_train: {vacias:,} de {celdas:,} celdas de medición están vacías ({100 * vacias / celdas:.1f} %)".replace(",", "."))

rachas = faltantes["Racha máxima X_train (días)"]
print(f"Columnas con tramos vacíos de 3 días o menos: {int((rachas <= 3).sum())}; con tramos de más de 30 días: {int((rachas > 30).sum())}")

print("Faltantes por hoja: promedio del % de vacíos de sus columnas, en cada parte")
display(tabla_simple(faltantes.groupby("Hoja")[["% faltante X_train", "% faltante X_val", "% faltante X_test"]].mean()
                     .sort_values("% faltante X_train", ascending=False), precision=1))

print("Las 10 columnas con más faltantes en X_train")
display(tabla_simple(faltantes.sort_values("% faltante X_train", ascending=False).head(10).rename_axis("Columna").reset_index(), precision=1))

print("Columnas con más de 40 % de faltantes en validación o prueba (disponibilidad que cae)")
cayeron = faltantes[(faltantes["% faltante X_val"] > 40) | (faltantes["% faltante X_test"] > 40)]
display(tabla_simple(cayeron.rename_axis("Columna").reset_index(), precision=1))

X_train: 5.259 de 222.176 celdas de medición están vacías (2.4 %)
Columnas con tramos vacíos de 3 días o menos: 204; con tramos de más de 30 días: 5
Faltantes por hoja: promedio del % de vacíos de sus columnas, en cada parte


,% faltante X_train,% faltante X_val,% faltante X_test
Hoja,,,
Totalizadores Glicol,8.6,1.6,0.5
Consolidado KPI,6.9,17.8,13.1
Totalizadores Produccion,6.3,1.6,0.6
Consolidado Produccion,3.3,1.6,0.5
Consolidado EE,2.5,3.2,1.7
Totalizadores Efluentes,0.7,1.6,0.5
Consolidado Aire,0.5,1.6,0.5
Totalizadores Gas y Vapor,0.3,1.7,0.5
Totalizadores Agua,0.3,1.7,0.6


Las 10 columnas con más faltantes en X_train


,Columna,Hoja,% faltante X_train,% faltante X_val,% faltante X_test,Racha máxima X_train (días)
0,Totalizadores Produccion | Nivel Silo Bagazo Norte (1),Totalizadores Produccion,36.6,1.6,1.6,308
1,Totalizadores Produccion | Nivel Silo Bagazo Sur (2),Totalizadores Produccion,36.6,1.6,0.5,308
2,Consolidado KPI | Aire L5 / Hl,Consolidado KPI,34.6,61.0,51.6,22
3,Consolidado KPI | Agua Linea 5/Hl,Consolidado KPI,34.6,61.0,51.6,22
4,Consolidado KPI | ET Linea 5/Hl,Consolidado KPI,34.6,61.0,51.6,22
5,Consolidado Produccion | Cocimientos Diarios,Consolidado Produccion,30.5,1.6,0.5,257
6,Totalizadores Produccion | Cocimientos Diarios,Totalizadores Produccion,30.5,1.6,0.5,257
7,Consolidado KPI | EE Linea 3 / Hl,Consolidado KPI,14.6,34.1,25.8,23
8,Consolidado KPI | ET Linea 3/Hl,Consolidado KPI,13.2,33.5,26.4,23
9,Consolidado KPI | Agua Linea 3/Hl,Consolidado KPI,13.1,33.5,25.3,23


Columnas con más de 40 % de faltantes en validación o prueba (disponibilidad que cae)


,Columna,Hoja,% faltante X_train,% faltante X_val,% faltante X_test,Racha máxima X_train (días)
0,Consolidado KPI | Agua Linea 5/Hl,Consolidado KPI,34.6,61.0,51.6,22
1,Consolidado KPI | ET Linea 5/Hl,Consolidado KPI,34.6,61.0,51.6,22
2,Consolidado KPI | Aire L5 / Hl,Consolidado KPI,34.6,61.0,51.6,22


### Paso 5: empaquetamos la limpieza como función reutilizable

Los pasos 1 a 3 quedaron programados en `src/limpieza.py` para que `predict.py` aplique la misma limpieza a datos nuevos sin reescribirla:

- `aprender_reglas_de_limpieza(X_train)` mide, **solo con entrenamiento**, qué columnas se conservan, cuáles son de energía y cuáles son los límites del percentil 99.
- `limpiar(tabla, reglas)` aplica esas reglas a cualquier tabla, sin recalcular nada. No borra filas ni rellena vacíos.

La celda siguiente parte de las tablas sin limpiar y comprueba que las funciones dan **exactamente** las mismas tres tablas que los pasos manuales de arriba. Las pruebas de las funciones están en `tests/test_limpieza.py`.

In [9]:
from limpieza import aprender_reglas_de_limpieza, limpiar

# Se vuelve a dividir la tabla completa para partir de las tablas sin limpiar
sin_limpiar_train, sin_limpiar_val, sin_limpiar_test = dividir_por_fecha(tabla_completa)

reglas = aprender_reglas_de_limpieza(sin_limpiar_train)          # solo con entrenamiento
X_train_funcion = limpiar(sin_limpiar_train, reglas)
X_val_funcion = limpiar(sin_limpiar_val, reglas)
X_test_funcion = limpiar(sin_limpiar_test, reglas)

print("True si las funciones dan el mismo resultado que el procedimiento manual")
display(tabla_simple(pd.DataFrame({"Igual": {
    "X_train": X_train.equals(X_train_funcion),
    "X_val": X_val.equals(X_val_funcion),
    "X_test": X_test.equals(X_test_funcion)}}), precision=0))

True si las funciones dan el mismo resultado que el procedimiento manual


,Igual
X_train,True
X_val,True
X_test,True
